# Chuyển `customer_turns` trong scenario JSON thành audio bằng Soniox

Notebook này đọc **một file JSON** có cấu trúc giống các file trong `scenarios_complete dataset`, lấy `calls.<call_id>.customer_turns`, rồi tạo **một file WAV cho mỗi cuộc gọi** bằng Soniox Realtime TTS.

Trước khi chạy:

1. Chạy ô cài đặt thư viện.
2. Thêm `SONIOX_API_KEY=...` vào `/content/.env`, hoặc đặt biến môi trường `SONIOX_API_KEY`. Không ghi API key trực tiếp vào notebook.
3. Chọn `SCENARIO_FILE`, kiểm tra phần xem trước, rồi đổi `RUN_TTS = True`.

> Soniox tính phí/có quota theo tài khoản. Mặc định `RUN_TTS = False` để việc Run All không vô tình gọi API.

In [ ]:
%pip install -q soniox python-dotenv

In [ ]:
import json
import os
import re
from pathlib import Path
from uuid import uuid4

from dotenv import load_dotenv
from IPython.display import Audio, display
from soniox import SonioxClient
from soniox.types import RealtimeTTSConfig

## 1. Chọn file đầu vào

Notebook dành cho Google Colab: mount Google Drive, sau đó đổi `SCENARIO_FILE` sang file bạn muốn xử lý. Các biến đường dẫn được giữ ở kiểu `Path` để có thể dùng `.is_file()`, toán tử `/` và các hàm ghi file ở những ô sau.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content")
DATASET_DIR = Path("/content/drive/MyDrive/scenarios_complete dataset")

# Thay tên file tại đây nếu cần.
SCENARIO_FILE = DATASET_DIR / "SC-001.json"
OUTPUT_DIR = Path("/content")

if not SCENARIO_FILE.is_file():
    raise FileNotFoundError(f"Không tìm thấy file: {SCENARIO_FILE}")

print("Input :", SCENARIO_FILE)
print("Output:", OUTPUT_DIR)

## 2. Đọc và kiểm tra `customer_turns`

In [ ]:
def load_customer_turns(json_path: Path) -> tuple[str, dict[str, list[str]]]:
    """Đọc scenario và trả về (scenario_id, {call_id: customer_turns})."""
    with json_path.open("r", encoding="utf-8") as file:
        payload = json.load(file)

    if not isinstance(payload, dict):
        raise ValueError("Nội dung JSON phải là một object.")

    scenario_id = str(payload.get("scenario_id") or json_path.stem)
    calls = payload.get("calls")
    if not isinstance(calls, dict) or not calls:
        raise ValueError("Không tìm thấy object 'calls' hoặc 'calls' đang rỗng.")

    turns_by_call: dict[str, list[str]] = {}
    for call_id, call_data in calls.items():
        if not isinstance(call_data, dict):
            raise ValueError(f"{call_id} phải là một object.")

        turns = call_data.get("customer_turns")
        if not isinstance(turns, list):
            raise ValueError(f"{call_id}.customer_turns phải là một list.")
        if not all(isinstance(turn, str) for turn in turns):
            raise ValueError(f"Mọi phần tử trong {call_id}.customer_turns phải là chuỗi.")

        cleaned_turns = [turn.strip() for turn in turns if turn.strip()]
        if not cleaned_turns:
            raise ValueError(f"{call_id}.customer_turns không có câu hợp lệ.")
        turns_by_call[str(call_id)] = cleaned_turns

    return scenario_id, turns_by_call


scenario_id, customer_turns_by_call = load_customer_turns(SCENARIO_FILE)
print(f"Scenario: {scenario_id}")
for call_id, turns in customer_turns_by_call.items():
    print(f"\n{call_id} ({len(turns)} câu):")
    for index, text in enumerate(turns, start=1):
        print(f"  {index}. {text}")

## 3. Cấu hình TTS

- `CALL_IDS = None`: xử lý tất cả các cuộc gọi. Ví dụ `CALL_IDS = ["call_1"]` để chỉ xử lý một cuộc gọi.
- `OVERWRITE = False`: bỏ qua WAV đã tồn tại.
- Dữ liệu chủ yếu là tiếng Việt nên `LANGUAGE = "vi"`. Voice Soniox có thể dùng xuyên ngôn ngữ.
- Mỗi phần tử của `customer_turns` được gửi như một text chunk, theo đúng thứ tự trong JSON.

In [ ]:
MODEL = "tts-rt-v2"
VOICE = "Daniel"
LANGUAGE = "vi"
SPEED = 1.0
RETURN_TIMESTAMPS = True

CALL_IDS = None  # None = tất cả; ví dụ: ["call_1"]
OVERWRITE = False
RUN_TTS = False  # Đổi thành True sau khi đã kiểm tra nội dung phía trên.

load_dotenv(PROJECT_ROOT / ".env")
SONIOX_API_KEY = os.getenv("SONIOX_API_KEY")
print("Đã tìm thấy SONIOX_API_KEY:", bool(SONIOX_API_KEY))

In [ ]:
def safe_filename(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", value).strip("._") or "audio"


def synthesize_call(
    client: SonioxClient,
    scenario_id: str,
    call_id: str,
    text_chunks: list[str],
    output_dir: Path,
) -> tuple[Path, int, str]:
    """Tạo một WAV chứa toàn bộ customer_turns của một call."""
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / (
        f"{safe_filename(scenario_id)}_{safe_filename(call_id)}_customer.wav"
    )

    config = RealtimeTTSConfig(
        stream_id=f"{safe_filename(scenario_id)}-{safe_filename(call_id)}-{uuid4()}",
        model=MODEL,
        voice=VOICE,
        language=LANGUAGE,
        speed=SPEED,
        return_timestamps=RETURN_TIMESTAMPS,
        audio_format="wav",
    )

    # Thêm khoảng trắng để hai text chunk không bị dính từ khi phát liên tục.
    chunks_for_tts = [f"{text.strip()} " for text in text_chunks]
    audio_chunks: list[bytes] = []

    with client.realtime.tts.connect(config=config) as session:
        session.send_text_chunks(chunks_for_tts, text_end=True)
        for chunk in session.receive_audio_chunks():
            audio_chunks.append(chunk)
        final_message = str(session.last_message)

    audio = b"".join(audio_chunks)
    if not audio:
        raise RuntimeError(f"Soniox không trả về audio cho {call_id}.")

    output_path.write_bytes(audio)
    return output_path, len(audio), final_message

## 4. Gọi Soniox và lưu WAV

Ô này chỉ gọi API khi `RUN_TTS = True`. Kết quả và manifest được lưu trong `tmp/soniox_tts/<scenario_id>/`.

In [ ]:
if not RUN_TTS:
    print("Chưa gọi API. Hãy đổi RUN_TTS = True trong ô cấu hình khi sẵn sàng.")
else:
    if not SONIOX_API_KEY:
        raise EnvironmentError(
            "Thiếu SONIOX_API_KEY. Hãy thêm key vào .env hoặc biến môi trường."
        )

    if CALL_IDS is None:
        selected_call_ids = list(customer_turns_by_call)
    else:
        selected_call_ids = list(dict.fromkeys(CALL_IDS))
        missing = [call_id for call_id in selected_call_ids if call_id not in customer_turns_by_call]
        if missing:
            raise KeyError(f"CALL_IDS không tồn tại trong file: {missing}")

    scenario_output_dir = OUTPUT_DIR / safe_filename(scenario_id)
    results = []
    generated_files = []
    client = SonioxClient(api_key=SONIOX_API_KEY)

    try:
        for call_id in selected_call_ids:
            expected_path = scenario_output_dir / (
                f"{safe_filename(scenario_id)}_{safe_filename(call_id)}_customer.wav"
            )
            if expected_path.exists() and not OVERWRITE:
                print(f"Bỏ qua file đã tồn tại: {expected_path.name}")
                results.append(
                    {
                        "call_id": call_id,
                        "status": "skipped_existing",
                        "output_file": str(expected_path),
                    }
                )
                continue

            try:
                output_path, byte_count, final_message = synthesize_call(
                    client=client,
                    scenario_id=scenario_id,
                    call_id=call_id,
                    text_chunks=customer_turns_by_call[call_id],
                    output_dir=scenario_output_dir,
                )
                generated_files.append(output_path)
                results.append(
                    {
                        "call_id": call_id,
                        "status": "generated",
                        "turn_count": len(customer_turns_by_call[call_id]),
                        "bytes": byte_count,
                        "output_file": str(output_path),
                        "final_message": final_message,
                    }
                )
                print(f"Đã ghi {byte_count:,} bytes: {output_path}")
            except Exception as exc:
                results.append(
                    {
                        "call_id": call_id,
                        "status": "error",
                        "error": f"{type(exc).__name__}: {exc}",
                    }
                )
                print(f"Lỗi ở {call_id}: {type(exc).__name__}: {exc}")
    finally:
        client.close()

    scenario_output_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = scenario_output_dir / "manifest.json"
    manifest = {
        "source_file": str(SCENARIO_FILE),
        "scenario_id": scenario_id,
        "model": MODEL,
        "voice": VOICE,
        "language": LANGUAGE,
        "speed": SPEED,
        "results": results,
    }
    manifest_path.write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print("Manifest:", manifest_path)

    if generated_files:
        print("Nghe thử file vừa tạo đầu tiên:")
        display(Audio(filename=str(generated_files[0])))